# Trích xuất dữ liệu từ file CoNLL (Data Extraction)

In [1]:
'''
Trích xuất câu và nhãn gold từ VTB-SRL CoNLL files.

Input:
    data/VTB-SRL/vtb-*.conll

Output:
    data/gold/1000_sentences.txt
    data/gold/gold_labels.json
'''

import os
import glob
import json

INPUT_DIR = os.path.join("data", "VTB-SRL")
OUTPUT_TXT = os.path.join("data", "gold", "vi_sentences.txt")
OUTPUT_JSON = os.path.join("data", "gold", "gold_labels.json")
TARGET_SENTENCE_COUNT = None  # None = lấy tất cả

# Cột số 2 (index 1) là cột chứa từ, cột số 13 (index 12) chứa cờ 'Y' (Vị từ)
WORD_COL_INDEX = 1
PREDICATE_COL_INDEX = 12


def process_conll_files():
    os.makedirs(os.path.dirname(OUTPUT_TXT), exist_ok=True)
    os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

    sentences_txt = []
    gold_labels_json = []

    file_pattern = os.path.join(INPUT_DIR, "vtb-*.conll")
    conll_files = sorted(glob.glob(file_pattern))

    if not conll_files:
        print(f"Không tìm thấy file nào ở đường dẫn '{file_pattern}'.")
        return

    print(f"Tìm thấy {len(conll_files)} file CoNLL.")
    sentence_count = 0

    for file_path in conll_files:
        if TARGET_SENTENCE_COUNT and sentence_count >= TARGET_SENTENCE_COUNT:
            break

        with open(file_path, 'r', encoding='utf-8') as f:
            current_sentence_lines = []

            for line in f:
                line = line.strip()

                if not line:
                    if current_sentence_lines:
                        has_verb = False
                        words = []
                        tokens_data = []

                        for row in current_sentence_lines:
                            cols = row.split('\t')

                            if len(cols) > WORD_COL_INDEX:
                                word = cols[WORD_COL_INDEX]
                                words.append(word)

                                is_predicate = False
                                if len(cols) > PREDICATE_COL_INDEX and cols[PREDICATE_COL_INDEX] == 'Y':
                                    has_verb = True
                                    is_predicate = True

                                srl_labels = cols[PREDICATE_COL_INDEX + 1:] if len(cols) > (PREDICATE_COL_INDEX + 1) else []

                                tokens_data.append({
                                    "word": word,
                                    "is_predicate": is_predicate,
                                    "labels": srl_labels
                                })

                        if has_verb:
                            full_sentence = " ".join(words)
                            sentences_txt.append(full_sentence)
                            gold_labels_json.append({
                                "id": sentence_count + 1,
                                "sentence": full_sentence,
                                "tokens": tokens_data
                            })
                            sentence_count += 1
                            if TARGET_SENTENCE_COUNT and sentence_count >= TARGET_SENTENCE_COUNT:
                                break

                    current_sentence_lines = []
                else:
                    current_sentence_lines.append(line)

    with open(OUTPUT_TXT, 'w', encoding='utf-8') as f_txt:
        for sent in sentences_txt:
            f_txt.write(sent + '\n')

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f_json:
        json.dump(gold_labels_json, f_json, ensure_ascii=False, indent=4)

    print(f"Đã trích xuất {sentence_count} câu.")
    print(f"File Text : {OUTPUT_TXT}")
    print(f"File JSON : {OUTPUT_JSON}")


if __name__ == "__main__":
    process_conll_files()

Tìm thấy 10 file CoNLL.
Đã trích xuất 923 câu.
File Text : data\gold\vi_sentences.txt
File JSON : data\gold\gold_labels.json


# Dịch Việt -> Anh (Translation)

In [2]:
'''
Dịch Việt -> Anh sử dụng Google Translate (deep-translator).
Lưu kèm original_idx để tránh lệch index khi có câu bị skip.

Input:
    data/gold/1000_sentences.txt

Output:
    data/translation/en_sentences.txt
    data/translation/translation_pair.json
    reports/translation_log.json
'''

import json
import time
import os
from pathlib import Path
from tqdm import tqdm
from deep_translator import GoogleTranslator


INPUT_FILE  = os.path.join("data", "gold", "vi_sentences.txt")
OUTPUT_TXT  = os.path.join("data", "translation", "en_sentences.txt")
OUTPUT_JSON = os.path.join("data", "translation", "translation_pair.json")
LOG_FILE    = os.path.join("reports", "translation_log.json")

BATCH_SIZE       = 10
DELAY_BATCH      = 1.5
DELAY_RETRY      = 5
MAX_RETRIES      = 3
CHECKPOINT_EVERY = 50


def load_vietnamese_sentences(path: str) -> list[str]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    sentences = [line.strip() for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"Đọc xong {len(sentences)} câu tiếng Việt từ '{path}'")
    return sentences


def load_checkpoint(log_path: str) -> dict:
    p = Path(log_path)
    if p.exists():
        data = json.loads(p.read_text(encoding="utf-8"))
        done = len([x for x in data.get("results", []) if x["en"] is not None])
        print(f"Tìm thấy checkpoint: đã dịch {done} câu — tiếp tục từ đây.")
        return data
    return {"results": [], "errors": []}


def save_checkpoint(log_path: str, state: dict):
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    Path(log_path).write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")


def translate_sentence(translator: GoogleTranslator, vi_text: str, idx: int) -> str | None:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            en = translator.translate(vi_text)
            if en:
                return en.strip()
        except Exception as e:
            print(f"  Câu {idx} – lần {attempt}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(DELAY_RETRY)
    print(f"  Câu {idx}: THẤT BẠI sau {MAX_RETRIES} lần thử.")
    return None


def translate_all(sentences: list[str]) -> list[dict]:
    translator = GoogleTranslator(source="vi", target="en")
    state      = load_checkpoint(LOG_FILE)

    done_ids = {r["idx"] for r in state["results"]}
    pending  = [(i, s) for i, s in enumerate(sentences) if i not in done_ids]
    print(f"Cần dịch thêm {len(pending)} câu (bỏ qua {len(done_ids)} câu đã có).")

    with tqdm(total=len(pending), desc="Dịch thuật", unit="câu") as pbar:
        for batch_start in range(0, len(pending), BATCH_SIZE):
            batch = pending[batch_start: batch_start + BATCH_SIZE]

            for idx, vi_text in batch:
                en_text = translate_sentence(translator, vi_text, idx)
                record  = {"idx": idx, "vi": vi_text, "en": en_text}
                state["results"].append(record)
                if en_text is None:
                    state["errors"].append(idx)
                pbar.update(1)

            time.sleep(DELAY_BATCH)

            if (batch_start // BATCH_SIZE + 1) % (CHECKPOINT_EVERY // BATCH_SIZE) == 0:
                save_checkpoint(LOG_FILE, state)
                print(f"  Checkpoint lưu tại câu ~{batch_start + BATCH_SIZE}")

    save_checkpoint(LOG_FILE, state)
    state["results"].sort(key=lambda x: x["idx"])
    return state["results"]


def save_outputs(results: list[dict]):
    os.makedirs(os.path.dirname(OUTPUT_TXT), exist_ok=True)
    os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

    pairs    = []
    en_lines = []
    failed   = []

    for r in results:
        if r["en"]:
            # Lưu kèm original_idx để các bước sau dùng để tránh lệch index
            pairs.append({"original_idx": r["idx"], "vi": r["vi"], "en": r["en"]})
            en_lines.append(r["en"])
        else:
            pairs.append({"original_idx": r["idx"], "vi": r["vi"], "en": "[TRANSLATION_FAILED]"})
            en_lines.append("[TRANSLATION_FAILED]")
            failed.append(r["idx"])

    Path(OUTPUT_TXT).write_text("\n".join(en_lines), encoding="utf-8")
    print(f"Đã lưu {len(en_lines)} câu tiếng Anh -> '{OUTPUT_TXT}'")

    Path(OUTPUT_JSON).write_text(
        json.dumps(pairs, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Đã lưu {len(pairs)} cặp câu -> '{OUTPUT_JSON}'")

    success_rate = (len(pairs) - len(failed)) / len(pairs) * 100 if pairs else 0
    print("=" * 50)
    print(f"TỔNG KẾT:")
    print(f"  Tổng câu            : {len(pairs)}")
    print(f"  Dịch thành công     : {len(pairs) - len(failed)}")
    print(f"  Thất bại            : {len(failed)}")
    print(f"  Tỉ lệ thành công    : {success_rate:.1f}%")
    if failed:
        print(f"  Câu thất bại (index): {failed}")
    print("=" * 50)


def main():
    print("DỊCH VIỆT -> ANH")
    sentences = load_vietnamese_sentences(INPUT_FILE)
    results   = translate_all(sentences)
    save_outputs(results)
    print("HOÀN THÀNH!")
    print(f"File TXT : '{OUTPUT_TXT}'")
    print(f"File JSON: '{OUTPUT_JSON}'")


if __name__ == "__main__":
    main()

DỊCH VIỆT -> ANH
Đọc xong 923 câu tiếng Việt từ 'data\gold\vi_sentences.txt'
Cần dịch thêm 923 câu (bỏ qua 0 câu đã có).


Dịch thuật:   5%|███▋                                                                | 50/923 [01:10<24:09,  1.66s/câu]

  Checkpoint lưu tại câu ~50


Dịch thuật:  11%|███████▎                                                           | 100/923 [02:14<12:44,  1.08câu/s]

  Checkpoint lưu tại câu ~100


...

  Checkpoint lưu tại câu ~850


Dịch thuật:  98%|█████████████████████████████████████████████████████████████████▎ | 900/923 [20:05<00:27,  1.20s/câu]

  Checkpoint lưu tại câu ~900


Dịch thuật: 100%|███████████████████████████████████████████████████████████████████| 923/923 [20:39<00:00,  1.34s/câu]

Đã lưu 923 câu tiếng Anh -> 'data\translation\en_sentences.txt'
Đã lưu 923 cặp câu -> 'data\translation\translation_pair.json'
TỔNG KẾT:
  Tổng câu            : 923
  Dịch thành công     : 923
  Thất bại            : 0
  Tỉ lệ thành công    : 100.0%
HOÀN THÀNH!
File TXT : 'data\translation\en_sentences.txt'
File JSON: 'data\translation\translation_pair.json'


# Gán nhãn SRL tiếng Anh (English SRL Tagging)

Gán nhãn SRL tiếng anh dùng thư viện allennlp (chỉ dùng được ở phiên bản 3.8) tách riêng ra ở file english_srl.ipynb

# Gióng hàng từ vựng (Word Alignment)

In [4]:
'''
Gióng hàng token EN <-> VI bằng SimAlign.
Fix:
  - Dùng original_idx thay positional index để tránh lệch câu
  - Chiến lược alignment: itermax primary, union fallback khi coverage thấp
  - Lọc câu có coverage quá thấp (bad translation filter)

Input:
    data/silver/english_labels_2.json
    data/gold/1000_sentences.txt

Output:
    reports/alignment_2.json
'''

import json
import os
from simalign import SentenceAligner
from underthesea import word_tokenize

EN_LABELS_FILE  = os.path.join("data", "silver", "english_labels_2.json")
VI_SENTENCES_FILE = os.path.join("data", "gold", "vi_sentences.txt")
ALIGNMENT_FILE  = os.path.join("reports", "alignment_2.json")

MIN_COVERAGE = 0.4   # Loại câu nếu coverage VI hoặc EN thấp hơn ngưỡng này


def load_vi_sentences(filepath):
    if not os.path.exists(filepath):
        print(f"ERROR: Không tìm thấy file {filepath}")
        return []
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def load_en_labels(filepath):
    if not os.path.exists(filepath):
        print(f"ERROR: Không tìm thấy file {filepath}")
        return []
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def tokenize_vietnamese(sentence):
    clean_sentence = sentence.replace("_", " ")
    return word_tokenize(clean_sentence, format="text").split()


def get_alignments(result, vi_tokens):
    """
    Chiến lược alignment:
    - Dùng itermax làm primary (balanced nhất giữa precision và recall)
    - Chỉ fallback sang UNION khi coverage VI < MIN_COVERAGE
    Tránh dùng UNION mặc định vì tăng nhiễu (precision thấp).
    """
    # Primary: itermax
    primary = [[pair[0], pair[1]] for pair in result.get("itermax", [])]
    vi_covered = len(set(p[1] for p in primary))

    if len(vi_tokens) > 0 and vi_covered / len(vi_tokens) < MIN_COVERAGE:
        # Fallback: union tất cả phương pháp
        seen = set()
        merged = []
        for method in ["argmax", "itermax", "match"]:
            for pair in result.get(method, []):
                key = (pair[0], pair[1])
                if key not in seen:
                    seen.add(key)
                    merged.append([pair[0], pair[1]])
        merged.sort(key=lambda x: (x[0], x[1]))
        return merged

    return primary


def is_alignment_acceptable(align_list, vi_tokens, en_tokens):
    """
    Kiểm tra coverage tối thiểu của alignment.
    Câu có coverage thấp thường do dịch sai cấu trúc -> loại bỏ thay vì giữ data nhiễu.
    """
    if not align_list:
        return False
    vi_covered = len(set(p[1] for p in align_list)) / len(vi_tokens)
    en_covered = len(set(p[0] for p in align_list)) / len(en_tokens)
    return vi_covered >= MIN_COVERAGE and en_covered >= MIN_COVERAGE


def main():
    en_data      = load_en_labels(EN_LABELS_FILE)
    vi_sentences = load_vi_sentences(VI_SENTENCES_FILE)

    if not en_data or not vi_sentences:
        print("ERROR: Dữ liệu đầu vào bị thiếu. Dừng chương trình.")
        return

    # Build map theo original_idx thay vì positional index
    # Tránh lệch câu khi SRL bỏ qua một số câu dịch thất bại
    en_map = {item["original_idx"]: item for item in en_data}

    print(f"VI sentences : {len(vi_sentences)}")
    print(f"EN labels    : {len(en_data)}")
    print(f"EN map keys  : {len(en_map)}")
    if len(vi_sentences) != len(en_data):
        print("CẢNH BÁO: Số câu VI và EN không bằng nhau — dùng original_idx để align đúng.")

    try:
        aligner = SentenceAligner(
            model="bert",
            token_type="bpe",
            matching_methods="aim"   # argmax + itermax + match
        )
    except Exception as e:
        print(f"ERROR: Khởi tạo SimAlign thất bại: {e}")
        return

    results        = []
    skipped_no_en  = 0
    skipped_bad_cov = 0
    skipped_error  = 0

    print(f"Bắt đầu gióng hàng cho {len(vi_sentences)} câu VI...")

    for i, vi_sent in enumerate(vi_sentences):
        # Lấy EN item theo original_idx
        if i not in en_map:
            skipped_no_en += 1
            continue

        en_item   = en_map[i]
        en_tokens = en_item.get("words", [])
        vi_tokens = tokenize_vietnamese(vi_sent)

        if not en_tokens or not vi_tokens:
            skipped_no_en += 1
            continue

        try:
            result     = aligner.get_word_aligns(en_tokens, vi_tokens)
            # pair format từ SimAlign: (en_idx, vi_idx)
            align_list = get_alignments(result, vi_tokens)
        except Exception as e:
            print(f"WARNING: Lỗi gióng hàng tại câu {i}. Chi tiết: {e}")
            skipped_error += 1
            continue

        # Lọc câu có coverage thấp (bad translation filter)
        if not is_alignment_acceptable(align_list, vi_tokens, en_tokens):
            skipped_bad_cov += 1
            continue

        vi_cov = len(set(p[1] for p in align_list)) / len(vi_tokens) * 100
        en_cov = len(set(p[0] for p in align_list)) / len(en_tokens) * 100

        results.append({
            "sent_id"  : i,          # = original_idx, dùng để lookup EN labels
            "vi_tokens": vi_tokens,
            "en_tokens": en_tokens,
            "pairs"    : align_list  # format: [en_idx, vi_idx]
        })

        if (i + 1) % 50 == 0:
            print(f"  [{i + 1}/{len(vi_sentences)}] VI_cov={vi_cov:.0f}% EN_cov={en_cov:.0f}% pairs={len(align_list)}")

    os.makedirs(os.path.dirname(ALIGNMENT_FILE), exist_ok=True)

    with open(ALIGNMENT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("=" * 50)
    print(f"Hoàn thành gióng hàng.")
    print(f"  Giữ lại          : {len(results)} câu")
    print(f"  Bỏ qua (no EN)   : {skipped_no_en} câu")
    print(f"  Bỏ qua (coverage): {skipped_bad_cov} câu")
    print(f"  Bỏ qua (lỗi)     : {skipped_error} câu")
    print(f"Kết quả lưu tại: {ALIGNMENT_FILE}")


if __name__ == "__main__":
    main()

VI sentences : 923
EN labels    : 923
EN map keys  : 923


Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3086.42it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-09 15:04:29,508 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
Ini

Bắt đầu gióng hàng cho 923 câu VI...
  [50/923] VI_cov=92% EN_cov=94% pairs=17
  [100/923] VI_cov=100% EN_cov=100% pairs=6
  [150/923] VI_cov=90% EN_cov=95% pairs=18
  [200/923] VI_cov=100% EN_cov=100% pairs=10
  [250/923] VI_cov=100% EN_cov=100% pairs=5
  [300/923] VI_cov=76% EN_cov=81% pairs=13
  [350/923] VI_cov=100% EN_cov=96% pairs=24
  [400/923] VI_cov=100% EN_cov=100% pairs=14
  [450/923] VI_cov=70% EN_cov=100% pairs=7
  [500/923] VI_cov=100% EN_cov=94% pairs=17
  [550/923] VI_cov=75% EN_cov=100% pairs=3
  [600/923] VI_cov=93% EN_cov=79% pairs=13
  [650/923] VI_cov=100% EN_cov=83% pairs=28
  [700/923] VI_cov=100% EN_cov=92% pairs=24
  [750/923] VI_cov=100% EN_cov=89% pairs=9
  [800/923] VI_cov=89% EN_cov=100% pairs=9
  [850/923] VI_cov=83% EN_cov=100% pairs=17
  [900/923] VI_cov=100% EN_cov=100% pairs=11
Hoàn thành gióng hàng.
  Giữ lại          : 922 câu
  Bỏ qua (no EN)   : 0 câu
  Bỏ qua (coverage): 1 câu
  Bỏ qua (lỗi)     : 0 câu
Kết quả lưu tại: reports\alignment_2.json


# Phóng chiếu nhãn (Label Projection)

In [5]:
'''
Project SRL labels từ EN sang VI qua alignment.
Fix:
  - Lookup EN labels theo original_idx (sent_id) thay positional index
  - Conflict resolution cải thiện: ưu tiên predicate, tie-break theo khoảng cách

Input:
    reports/alignment_2.json
    data/silver/english_labels_2.json

Output:
    data/silver/silver_raw_2.json
'''

import json
import os
from collections import defaultdict, Counter

ALIGNMENT_FILE = os.path.join("reports", "alignment_2.json")
EN_LABELS_FILE = os.path.join("data", "silver", "english_labels_2.json")
RAW_OUTPUT     = os.path.join("data", "silver", "silver_raw_2.json")


def normalize_label(bio_label):
    if bio_label == "O":
        return "_"
    raw = bio_label[2:] if bio_label.startswith(("B-", "I-")) else bio_label
    if raw == "V":
        return "V"
    if raw.startswith("ARGM-"):
        return "AM-" + raw[5:]
    if raw.startswith("ARG"):
        return "Arg" + raw[3:]
    return raw


def convert_bio_to_token_dicts(en_item):
    words = en_item.get("words", [])
    verbs = en_item.get("verbs", [])

    if not words or not verbs:
        return []

    num_cols = len(verbs)
    tokens = [{"word": w, "is_predicate": False, "labels": ["_"] * num_cols} for w in words]

    for col_idx, verb_info in enumerate(verbs):
        verb_name = verb_info.get("verb", "unknown")
        tags = verb_info.get("tags", [])

        for tok_idx, tag in enumerate(tags):
            if tok_idx >= len(tokens):
                break
            if tag in ("B-V", "I-V"):
                tokens[tok_idx]["labels"][col_idx] = f"{verb_name}.01"
                tokens[tok_idx]["is_predicate"] = True
            elif tag != "O":
                tokens[tok_idx]["labels"][col_idx] = normalize_label(tag)

    return tokens


def build_vi_to_en_map(pairs):
    """
    pairs format: [en_idx, vi_idx]  (SimAlign src=EN, tgt=VI)
    """
    vi_to_en = defaultdict(list)
    for pair in pairs:
        en_idx, vi_idx = int(pair[0]), int(pair[1])
        vi_to_en[vi_idx].append(en_idx)
    return vi_to_en


def find_predicate_en_idx(en_token_dicts, col_idx):
    """Tìm EN token index của predicate trong cột col_idx."""
    for idx, tok in enumerate(en_token_dicts):
        lbl = tok["labels"][col_idx] if col_idx < len(tok["labels"]) else "_"
        if lbl != "_" and ("." in lbl or lbl == "V"):
            return idx
    return None


def get_label(vi_idx, col_idx, vi_to_en, en_token_dicts, predicate_en_idx=None):
    en_indices = vi_to_en.get(vi_idx, [])
    if not en_indices:
        return "_"

    candidates = []
    for en_idx in en_indices:
        if en_idx < len(en_token_dicts):
            labels = en_token_dicts[en_idx].get("labels", [])
            if col_idx < len(labels):
                candidates.append((en_idx, labels[col_idx]))

    if not candidates:
        return "_"

    non_empty = [(idx, l) for idx, l in candidates if l != "_"]
    if not non_empty:
        return "_"
    if len(non_empty) == 1:
        return non_empty[0][1]

    # Ưu tiên predicate label (có dấu chấm hoặc là "V")
    pred_labels = [l for _, l in non_empty if "." in l or l == "V"]
    if pred_labels:
        return pred_labels[0]

    # Majority vote
    counts    = Counter(l for _, l in non_empty)
    max_count = max(counts.values())
    top       = [l for l, c in counts.items() if c == max_count]
    if len(top) == 1:
        return top[0]

    # Tie-break: chọn EN token gần predicate nhất
    if predicate_en_idx is not None:
        best = min(non_empty, key=lambda x: abs(x[0] - predicate_en_idx))
        return best[1]

    return top[0]


def main():
    print("Label Projection")

    if not os.path.exists(ALIGNMENT_FILE) or not os.path.exists(EN_LABELS_FILE):
        print("ERROR: Không tìm thấy file dữ liệu đầu vào.")
        return

    with open(ALIGNMENT_FILE, "r", encoding="utf-8") as f:
        alignment_data = json.load(f)
    with open(EN_LABELS_FILE, "r", encoding="utf-8") as f:
        en_data = json.load(f)

    # Build map theo original_idx để tránh lệch index
    en_map = {item["original_idx"]: item for item in en_data}
    print(f"EN labels map: {len(en_map)} câu")
    print(f"Alignment   : {len(alignment_data)} câu")

    silver_raw = []
    skipped    = 0

    for item in alignment_data:
        sent_id   = item.get("sent_id")   # = original_idx
        vi_tokens = item.get("vi_tokens", [])
        pairs     = item.get("pairs", [])

        # Lookup theo original_idx
        if sent_id not in en_map:
            skipped += 1
            continue

        en_token_dicts = convert_bio_to_token_dicts(en_map[sent_id])
        if not en_token_dicts:
            skipped += 1
            continue

        num_cols = len(en_token_dicts[0]["labels"])
        if num_cols == 0:
            skipped += 1
            continue

        vi_to_en       = build_vi_to_en_map(pairs)
        result_tokens  = []

        for vi_idx, word in enumerate(vi_tokens):
            labels = []
            for col in range(num_cols):
                pred_en_idx = find_predicate_en_idx(en_token_dicts, col)
                lbl = get_label(vi_idx, col, vi_to_en, en_token_dicts, pred_en_idx)
                labels.append(lbl)

            is_predicate = any(l != "_" and ("." in l or l == "V") for l in labels)
            result_tokens.append({
                "word"        : word,
                "is_predicate": is_predicate,
                "labels"      : labels
            })

        silver_raw.append({
            "id"         : sent_id + 1,
            "sentence_vi": " ".join(vi_tokens),
            "tokens"     : result_tokens
        })

    os.makedirs(os.path.dirname(RAW_OUTPUT), exist_ok=True)

    with open(RAW_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(silver_raw, f, ensure_ascii=False, indent=4)

    print("Phóng chiếu nhãn thành công.")
    print(f"Đã xử lý: {len(silver_raw)} câu. Bỏ qua: {skipped} câu.")
    print(f"Kết quả lưu tại: {RAW_OUTPUT}")


if __name__ == "__main__":
    main()

Label Projection
EN labels map: 923 câu
Alignment   : 922 câu
Phóng chiếu nhãn thành công.
Đã xử lý: 907 câu. Bỏ qua: 15 câu.
Kết quả lưu tại: data\silver\silver_raw_2.json


# Lọc bằng từ loại (POS Filtering)

In [6]:
'''
Lọc nhãn predicate sai bằng POS tagging tiếng Việt (underthesea).
Fix: Bỏ "N" và "V" ra khỏi BLOCKED_POS vì underthesea accuracy ~85%,
     tiếng Việt có danh động từ làm vị ngữ rất phổ biến.

Input:
    data/silver/silver_raw_2.json

Output:
    data/silver/silver_filtered_2.json
'''

import json
import os
from underthesea import pos_tag

RAW_INPUT      = os.path.join("data", "silver", "silver_raw_2.json")
CLEANED_OUTPUT = os.path.join("data", "silver", "silver_filtered_2.json")

# Bỏ "N" và "V" so với phiên bản cũ:
#   - "V" rõ ràng là động từ, không nên block
#   - "N" tiếng Việt hay dùng làm vị ngữ (danh động từ), underthesea đôi khi tag sai
BLOCKED_POS = {"Np", "M", "P", "E", "C", "T", "L", "CH", "Nc", "Nu"}


def is_predicate_label(label):
    if not label or label == "_":
        return False
    return "." in label or label == "V" or label == "null"


def get_pos_map(vi_tokens):
    try:
        tagged = pos_tag(" ".join(vi_tokens))
        return {i: tagged[i][1] for i in range(min(len(tagged), len(vi_tokens)))}
    except Exception:
        return {}


def find_predicate_idx(tokens, col_idx):
    for idx, tok in enumerate(tokens):
        if col_idx < len(tok.get("labels", [])) and is_predicate_label(tok["labels"][col_idx]):
            return idx
    return None


def filter_sentence(item):
    tokens    = item.get("tokens", [])
    vi_tokens = [t["word"] for t in tokens]

    if not tokens:
        return None

    num_cols = len(tokens[0]["labels"])
    if num_cols == 0:
        return None

    pos_map    = get_pos_map(vi_tokens)
    valid_cols = []

    for col_idx in range(num_cols):
        pred_idx = find_predicate_idx(tokens, col_idx)
        if pred_idx is None:
            continue

        pos = pos_map.get(pred_idx, "X")

        if pos not in BLOCKED_POS:
            valid_cols.append(col_idx)

    if not valid_cols:
        return None

    for tok in tokens:
        tok["labels"]       = [tok["labels"][i] for i in valid_cols]
        tok["is_predicate"] = any(is_predicate_label(l) for l in tok["labels"])

    return item


def main():
    print("POS Filtering - Underthesea")

    if not os.path.exists(RAW_INPUT):
        print(f"ERROR: Không tìm thấy file {RAW_INPUT}")
        return

    with open(RAW_INPUT, "r", encoding="utf-8") as f:
        silver_raw = json.load(f)

    filtered_data = []
    dropped_count = 0
    total         = len(silver_raw)

    print(f"Đang lọc {total} câu...")

    for i, item in enumerate(silver_raw):
        filtered_item = filter_sentence(item)
        if filtered_item is not None:
            filtered_data.append(filtered_item)
        else:
            dropped_count += 1

        if (i + 1) % 100 == 0:
            print(f"  [{i + 1}/{total}] Giữ: {len(filtered_data)} | Loại: {dropped_count}")

    # Đánh lại ID liên tục
    for new_id, item in enumerate(filtered_data, start=1):
        item["id"] = new_id

    os.makedirs(os.path.dirname(CLEANED_OUTPUT), exist_ok=True)

    with open(CLEANED_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(filtered_data, f, ensure_ascii=False, indent=4)

    print("Lọc POS thành công.")
    print(f"  Tổng ban đầu : {total}")
    print(f"  Giữ lại      : {len(filtered_data)} ({len(filtered_data)/total*100:.1f}%)")
    print(f"  Loại bỏ      : {dropped_count} ({dropped_count/total*100:.1f}%)")
    print(f"Kết quả lưu tại: {CLEANED_OUTPUT}")


if __name__ == "__main__":
    main()

POS Filtering - Underthesea
Đang lọc 907 câu...
  [100/907] Giữ: 93 | Loại: 7
  [200/907] Giữ: 190 | Loại: 10
  [300/907] Giữ: 287 | Loại: 13
  [400/907] Giữ: 384 | Loại: 16
  [500/907] Giữ: 478 | Loại: 22
  [600/907] Giữ: 574 | Loại: 26
  [700/907] Giữ: 669 | Loại: 31
  [800/907] Giữ: 767 | Loại: 33
  [900/907] Giữ: 858 | Loại: 42
Lọc POS thành công.
  Tổng ban đầu : 907
  Giữ lại      : 865 (95.4%)
  Loại bỏ      : 42 (4.6%)
Kết quả lưu tại: data\silver\silver_filtered_2.json


# Đánh giá F1

In [7]:
'''
Đánh giá F1 Silver vs Gold (token-aligned).

Input:
    data/silver/silver_filtered_2.json
    data/gold/gold_labels.json

Output:
    reports/evaluation_report_v2.json
'''

import json
import os
import re
from collections import defaultdict, Counter

SILVER_FILE = os.path.join("data", "silver", "silver_filtered_2.json")
GOLD_FILE   = os.path.join("data", "gold", "gold_labels.json")
REPORT_FILE = os.path.join("reports", "evaluation_report_v2.json")


def flatten(word):
    return word.replace("_", " ").lower().strip()


def sentence_key(tokens):
    return " ".join(flatten(t["word"]) for t in tokens)


def normalize_label(label):
    """
    Chuẩn hóa nhãn để so sánh công bằng giữa gold (VTB) và silver (PropBank).
    VTB và PropBank đều dùng Arg0/Arg1/AM-* nên chủ yếu cần lowercase + strip.
    """
    if not label or label == "_":
        return "_"
    lbl = label.strip().lower()

    # Bỏ phần sense number: "run.01" -> "run" (predicate matching)
    if re.match(r'.+\.\d+$', lbl):
        return lbl  # giữ nguyên cho predicate

    # AM-* variants (argm-, am-)
    if lbl.startswith("argm-"):
        return "am-" + lbl[5:]
    if lbl.startswith("am-"):
        return lbl

    # ARG0-of, ARG1-of dạng light verb -> strip "-of"
    m = re.match(r'^arg(\d)-of$', lbl)
    if m:
        return f"arg{m.group(1)}"

    # ARG0, ARG1, ... -> arg0, arg1
    m = re.match(r'^arg(\d)', lbl)
    if m:
        return f"arg{m.group(1)}"

    return lbl


def align_tokens(gold_tokens, silver_tokens):
    """
    Align token gold <-> silver dựa trên chuỗi ký tự.
    Trả về dict {silver_idx: [gold_idx, ...]}
    """
    silver_to_gold = defaultdict(list)
    gold_flat   = [flatten(t["word"]) for t in gold_tokens]
    silver_flat = [flatten(t["word"]) for t in silver_tokens]

    g_ptr = 0
    for s_idx, s_word in enumerate(silver_flat):
        s_chars = s_word.replace(" ", "")
        matched_chars = ""
        while g_ptr < len(gold_flat) and len(matched_chars) < len(s_chars):
            matched_chars += gold_flat[g_ptr].replace(" ", "")
            silver_to_gold[s_idx].append(g_ptr)
            g_ptr += 1
            if matched_chars == s_chars:
                break

    return silver_to_gold


def build_sentence_map(data):
    smap = {}
    for item in data:
        tokens = item.get("tokens", [])
        key    = sentence_key(tokens)
        smap[key] = item
    return smap


def compute_prf(tp, fp, fn):
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return round(p, 4), round(r, 4), round(f1, 4)


def evaluate(silver_data, gold_data):
    gold_map   = build_sentence_map(gold_data)
    silver_map = build_sentence_map(silver_data)

    common_keys = set(gold_map.keys()) & set(silver_map.keys())
    print(f"  Tổng câu Gold   : {len(gold_map)}")
    print(f"  Tổng câu Silver : {len(silver_map)}")
    print(f"  Câu khớp        : {len(common_keys)}\n")

    if not common_keys:
        print("ERROR: Không ghép khớp được câu nào.")
        return None

    pred_tp = pred_fp = pred_fn = 0
    arg_tp  = arg_fp  = arg_fn  = 0
    per_label       = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    sentence_stats  = []

    for key in sorted(common_keys):
        gold_item   = gold_map[key]
        silver_item = silver_map[key]

        gold_tokens   = gold_item["tokens"]
        silver_tokens = silver_item["tokens"]

        num_gold_cols   = len(gold_tokens[0].get("labels", [])) if gold_tokens else 0
        num_silver_cols = len(silver_tokens[0].get("labels", [])) if silver_tokens else 0

        if num_gold_cols == 0 or num_silver_cols == 0:
            continue

        silver_to_gold = align_tokens(gold_tokens, silver_tokens)

        # 1. Predicate detection
        silver_pred_gold_indices = set()
        for s_idx, tok in enumerate(silver_tokens):
            if tok.get("is_predicate"):
                for g_idx in silver_to_gold.get(s_idx, []):
                    silver_pred_gold_indices.add(g_idx)

        gold_pred_indices = {g_idx for g_idx, tok in enumerate(gold_tokens) if tok.get("is_predicate")}

        s_pred_tp = len(silver_pred_gold_indices & gold_pred_indices)
        s_pred_fp = len(silver_pred_gold_indices - gold_pred_indices)
        s_pred_fn = len(gold_pred_indices - silver_pred_gold_indices)

        pred_tp += s_pred_tp
        pred_fp += s_pred_fp
        pred_fn += s_pred_fn

        # 2. Argument labeling
        silver_arg_set = set()
        gold_arg_set   = set()

        for g_idx, tok in enumerate(gold_tokens):
            for lbl in tok.get("labels", []):
                norm = normalize_label(lbl)
                if norm != "_" and (norm.startswith("arg") or norm.startswith("am-")):
                    gold_arg_set.add((g_idx, norm))

        for s_idx, tok in enumerate(silver_tokens):
            gold_indices = silver_to_gold.get(s_idx, [])
            for lbl in tok.get("labels", []):
                norm = normalize_label(lbl)
                if norm != "_" and (norm.startswith("arg") or norm.startswith("am-")):
                    for g_idx in gold_indices:
                        silver_arg_set.add((g_idx, norm))

        s_arg_tp = len(silver_arg_set & gold_arg_set)
        s_arg_fp = len(silver_arg_set - gold_arg_set)
        s_arg_fn = len(gold_arg_set - silver_arg_set)

        arg_tp += s_arg_tp
        arg_fp += s_arg_fp
        arg_fn += s_arg_fn

        all_labels = set(l for _, l in gold_arg_set) | set(l for _, l in silver_arg_set)
        for lbl in all_labels:
            s_set = {i for i, l in silver_arg_set if l == lbl}
            g_set = {i for i, l in gold_arg_set if l == lbl}
            per_label[lbl]["tp"] += len(s_set & g_set)
            per_label[lbl]["fp"] += len(s_set - g_set)
            per_label[lbl]["fn"] += len(g_set - s_set)

        _, _, s_f1 = compute_prf(s_arg_tp, s_arg_fp, s_arg_fn)
        sentence_stats.append({
            "id"            : gold_item.get("id"),
            "sentence"      : gold_item.get("sentence", "")[:70],
            "gold_tokens"   : len(gold_tokens),
            "silver_tokens" : len(silver_tokens),
            "pred_tp": s_pred_tp, "pred_fp": s_pred_fp, "pred_fn": s_pred_fn,
            "arg_tp" : s_arg_tp,  "arg_fp" : s_arg_fp,  "arg_fn" : s_arg_fn,
            "arg_f1" : s_f1
        })

    pred_p, pred_r, pred_f1 = compute_prf(pred_tp, pred_fp, pred_fn)
    arg_p,  arg_r,  arg_f1  = compute_prf(arg_tp, arg_fp, arg_fn)
    overall_f1 = (pred_f1 + arg_f1) / 2

    per_label_results = {}
    for lbl, c in sorted(per_label.items()):
        p, r, f1 = compute_prf(c["tp"], c["fp"], c["fn"])
        per_label_results[lbl] = {
            "precision": p, "recall": r, "f1": f1,
            "tp": c["tp"], "fp": c["fp"], "fn": c["fn"]
        }

    return {
        "summary": {
            "n_gold"    : len(gold_map),
            "n_silver"  : len(silver_map),
            "n_matched" : len(common_keys),
            "n_evaluated": len(sentence_stats)
        },
        "predicate_detection": {
            "precision": pred_p, "recall": pred_r, "f1": pred_f1,
            "tp": pred_tp, "fp": pred_fp, "fn": pred_fn
        },
        "argument_labeling": {
            "precision": arg_p, "recall": arg_r, "f1": arg_f1,
            "tp": arg_tp, "fp": arg_fp, "fn": arg_fn
        },
        "overall_f1"    : overall_f1,
        "per_label"     : per_label_results,
        "sentence_stats": sentence_stats
    }


def print_results(r):
    print("=" * 60)
    print("KẾT QUẢ ĐÁNH GIÁ (F1 - Token-Aligned)")
    print("=" * 60)

    s = r["summary"]
    print(f"Gold    : {s['n_gold']} câu")
    print(f"Silver  : {s['n_silver']} câu")
    print(f"Match   : {s['n_matched']} câu")
    print(f"Đánh giá: {s['n_evaluated']} câu\n")

    pd = r["predicate_detection"]
    print("PHÁT HIỆN VỊ TỪ (PREDICATE DETECTION)")
    print(f"  P={pd['precision']:.4f}  R={pd['recall']:.4f}  F1={pd['f1']:.4f}"
          f"  (TP={pd['tp']} FP={pd['fp']} FN={pd['fn']})\n")

    al = r["argument_labeling"]
    print("GÁN NHÃN THAM SỐ (ARGUMENT LABELING)")
    print(f"  P={al['precision']:.4f}  R={al['recall']:.4f}  F1={al['f1']:.4f}"
          f"  (TP={al['tp']} FP={al['fp']} FN={al['fn']})\n")

    print(f"OVERALL F1: {r['overall_f1']:.4f}\n")

    print("F1 THEO TỪNG LOẠI NHÃN:")
    print(f"  {'Nhãn':<15} {'P':>7} {'R':>7} {'F1':>7} {'TP':>5} {'FP':>5} {'FN':>5}")
    print(f"  {'-' * 57}")
    for lbl, m in sorted(r["per_label"].items(), key=lambda x: -x[1]["f1"]):
        print(f"  {lbl:<15} {m['precision']:>7.4f} {m['recall']:>7.4f}"
              f" {m['f1']:>7.4f} {m['tp']:>5} {m['fp']:>5} {m['fn']:>5}")

    print("\n10 CÂU CÓ F1 THẤP NHẤT:")
    for s in sorted(r["sentence_stats"], key=lambda x: x["arg_f1"])[:10]:
        print(f"  id={str(s['id']):>4} f1={s['arg_f1']:.3f}"
              f" [G={s['gold_tokens']} S={s['silver_tokens']}] {s['sentence'][:55]}")
    print("=" * 60)


def main():
    print("Đang đọc dữ liệu...")

    if not os.path.exists(SILVER_FILE):
        print(f"ERROR: Không tìm thấy file Silver ({SILVER_FILE})")
        return
    if not os.path.exists(GOLD_FILE):
        print(f"ERROR: Không tìm thấy file Gold ({GOLD_FILE})")
        return

    with open(SILVER_FILE, "r", encoding="utf-8") as f:
        silver_data = json.load(f)
    with open(GOLD_FILE, "r", encoding="utf-8") as f:
        gold_data = json.load(f)

    print("Đang tính F1 Score...")
    results = evaluate(silver_data, gold_data)

    if results:
        print_results(results)
        os.makedirs(os.path.dirname(REPORT_FILE), exist_ok=True)
        with open(REPORT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"\nBáo cáo lưu tại: {REPORT_FILE}")


if __name__ == "__main__":
    main()

Đang đọc dữ liệu...
Đang tính F1 Score...
  Tổng câu Gold   : 922
  Tổng câu Silver : 864
  Câu khớp        : 848

KẾT QUẢ ĐÁNH GIÁ (F1 - Token-Aligned)
Gold    : 922 câu
Silver  : 864 câu
Match   : 848 câu
Đánh giá: 848 câu

PHÁT HIỆN VỊ TỪ (PREDICATE DETECTION)
  P=0.5554  R=0.6332  F1=0.5918  (TP=1193 FP=955 FN=691)

GÁN NHÃN THAM SỐ (ARGUMENT LABELING)
  P=0.5410  R=0.4072  F1=0.4646  (TP=5032 FP=4270 FN=7327)

OVERALL F1: 0.5282

F1 THEO TỪNG LOẠI NHÃN:
  Nhãn                  P       R      F1    TP    FP    FN
  ---------------------------------------------------------
  am-neg           0.7158  0.5440  0.6182    68    27    57
  arg1             0.6481  0.5715  0.6074  2542  1380  1906
  arg0             0.7481  0.4249  0.5420  1004   338  1359
  am-tmp           0.6372  0.4545  0.5306   425   242   510
  am-cau           0.7229  0.3550  0.4762    60    23   109
  am-prp           0.6073  0.3858  0.4718   201   130   320
  am-com           0.8000  0.3077  0.4444     8     2    